In [1]:
from pathlib import Path
import sys
import zipfile

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.feature_engineering import prepare_features

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
SUBMISSION_DIR = PROJECT_ROOT / "submission"

MODELS_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

In [2]:
train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

TARGET_COLUMN = "Цена"

X_train_features = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_test_features = prepare_features(X_test)

y = train[TARGET_COLUMN].copy()

In [3]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

feature_columns = [
    column
    for column in X_train_features.columns
    if column not in EXCLUDED_COLUMNS
]

numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

categorical_columns = [
    column
    for column in feature_columns
    if column not in numeric_columns
]

X_train_cat = X_train_features[feature_columns].copy()
X_test_cat = X_test_features[feature_columns].copy()

for column in categorical_columns:
    X_train_cat[column] = (
        X_train_cat[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_cat[column] = (
        X_test_cat[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert list(X_train_cat.columns) == list(X_test_cat.columns)
assert len(X_train_cat) == 8340
assert len(X_test_cat) == 8341
assert "car_id" not in X_train_cat.columns
assert "Предложение" not in X_train_cat.columns

In [4]:
final_catboost_3000 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

final_catboost_3000.fit(
    X_train_cat,
    np.log1p(y),
    cat_features=categorical_columns,
)

0:	learn: 0.6518951	total: 223ms	remaining: 11m 9s
300:	learn: 0.1830202	total: 27.3s	remaining: 4m 4s
600:	learn: 0.1497725	total: 56.1s	remaining: 3m 44s
900:	learn: 0.1297234	total: 1m 23s	remaining: 3m 14s
1200:	learn: 0.1146269	total: 1m 53s	remaining: 2m 49s
1500:	learn: 0.1020465	total: 2m 18s	remaining: 2m 18s
1800:	learn: 0.0921410	total: 2m 43s	remaining: 1m 48s
2100:	learn: 0.0835175	total: 3m 7s	remaining: 1m 20s
2400:	learn: 0.0763421	total: 3m 32s	remaining: 53s
2700:	learn: 0.0698097	total: 3m 56s	remaining: 26.2s
2999:	learn: 0.0639257	total: 4m 20s	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=8, iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=300)

In [5]:
test_predictions = np.expm1(
    final_catboost_3000.predict(X_test_cat)
)

test_predictions = np.maximum(test_predictions, 1)

print(pd.Series(test_predictions).describe())
print("NaN:", np.isnan(test_predictions).sum())
print("Inf:", np.isinf(test_predictions).sum())
print("<= 0:", (test_predictions <= 0).sum())

count      8341.000000
mean      36249.621912
std       29505.082975
min        4039.768421
25%       19111.688428
50%       29497.568140
75%       44255.080249
max      533635.384763
dtype: float64
NaN: 0
Inf: 0
<= 0: 0


In [6]:
submission = pd.DataFrame(
    {
        "Цена": test_predictions,
    }
)

assert submission.shape == (8341, 1)
assert submission["Цена"].notna().all()
assert np.isfinite(submission["Цена"]).all()
assert (submission["Цена"] > 0).all()

submission_path = SUBMISSION_DIR / "submission_catboost_3000.csv"

submission.to_csv(
    submission_path,
    index=False,
)

In [7]:
model_path = MODELS_DIR / "catboost_log1p_3000.joblib"

joblib.dump(
    final_catboost_3000,
    model_path,
)

zip_path = SUBMISSION_DIR / "submission_catboost_3000.zip"

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zipf:
    zipf.write(
        submission_path,
        arcname="submission.csv",
    )

with zipfile.ZipFile(zip_path, "r") as zipf:
    print(zipf.namelist())

['submission.csv']


In [8]:
check_submission = pd.read_csv(submission_path)

print(check_submission.shape)
print(check_submission.columns.tolist())
display(check_submission.head())

(8341, 1)
['Цена']


,Цена
0,31688.466520
1,27077.731220
2,39655.330613
3,57967.369213
4,26499.015677
